# Module 02 — Optimizer Rules & Join Strategies

Column pruning, filter pushdown, equi vs inequi joins, and how to read join nodes in `explain`.

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

from pyspark.sql import functions as F
spark = begin_module('module-02-plan-joins')
ui_links()

In [ ]:
trips = spark.read.parquet(*config.yellow_taxi_paths())
zones = spark.read.option('header', True).csv(config.zones_path())
trips_small = trips.select(
    'VendorID', 'PULocationID', 'DOLocationID', 'total_amount', 'tpep_pickup_datetime'
)

## Column pruning

In [ ]:
wide = trips  # many columns
narrow = trips.select('VendorID', 'total_amount')
narrow.explain('formatted')  # fewer Parquet columns read
wide.filter('total_amount > 80').select('VendorID').explain('formatted')

## Filter pushdown

In [ ]:
q = trips_small.filter('total_amount > 100 AND VendorID = 2')
q.explain('formatted')  # PushedFilters / PostScan filters on Parquet

## Equi-join baseline (sort-merge at scale)

In [ ]:
reset_spark_configs(spark, {
    'spark.sql.autoBroadcastJoinThreshold': '-1',
    'spark.sql.join.preferSortMergeJoin': 'true',
})
equi = trips_small.alias('t').join(
    trips_small.alias('s'),
    (F.col('t.VendorID') == F.col('s.VendorID'))
    & (F.col('t.PULocationID') == F.col('s.PULocationID')),
)
equi.select('t.VendorID', 't.PULocationID').limit(5).explain('formatted')

## Broadcast hash join (small dimension)

In [ ]:
z = zones.withColumnRenamed('LocationID', 'PULocationID')
reset_spark_configs(spark, {'spark.sql.autoBroadcastJoinThreshold': '52428800'})
bj = trips_small.join(z, 'PULocationID', 'inner')
bj.explain('formatted')  # BroadcastExchange / BroadcastHashJoin

## Shuffle hash join (disable sort-merge preference)

In [ ]:
reset_spark_configs(spark, {
    'spark.sql.autoBroadcastJoinThreshold': '-1',
    'spark.sql.join.preferSortMergeJoin': 'false',
})
big = trips_small.repartition(32, 'VendorID')
shj = big.alias('a').join(big.alias('b'), 'VendorID', 'inner')
shj.limit(1).explain('formatted')  # ShuffledHashJoin when chosen

## Sort-merge join (default for large equi)

In [ ]:
reset_spark_configs(spark, {
    'spark.sql.autoBroadcastJoinThreshold': '-1',
    'spark.sql.join.preferSortMergeJoin': 'true',
})
smj = big.alias('a').join(big.alias('b'), 'PULocationID', 'inner')
smj.limit(1).explain('formatted')  # SortMergeJoin

## Cartesian / broadcast nested loop (dangerous at scale)

In [ ]:
tiny_a = trips_small.select('VendorID').distinct().limit(5)
tiny_b = trips_small.select('PULocationID').distinct().limit(5)
cross = tiny_a.crossJoin(tiny_b)
cross.explain('formatted')
spark.conf.set('spark.sql.crossJoin.enabled', 'true')

In [ ]:
inequi_loop = tiny_a.alias('a').join(
    tiny_b.alias('b'),
    F.col('a.VendorID') < F.col('b.PULocationID'),
)
inequi_loop.explain('formatted')  # often BroadcastNestedLoopJoin or Cartesian

## Inequi join challenge — decompose to equi + filter

Predicate: match trips whose `total_amount` is within $5 (range / inequi).
Instead of `abs(t-s) < 5` in the join, **bucket** amounts and join on equi keys, then filter.

In [ ]:
a = trips_small.select('VendorID', 'total_amount').withColumn(
    'amt_bucket', F.floor(F.col('total_amount') / 5)
)
b = a.select(
    F.col('VendorID').alias('v2'),
    F.col('total_amount').alias('amt_b'),
    F.col('amt_bucket').alias('b_bucket'),
)
equi_buckets = a.join(
    b,
    (F.col('VendorID') == F.col('v2')) & (F.col('amt_bucket') == F.col('b_bucket')),
)
refined = equi_buckets.filter(
    F.abs(F.col('total_amount') - F.col('amt_b')) <= 5
)
refined.limit(5).explain('formatted')
print('rows (capped action):', refined.limit(1000).count())

## Takeaways

1. Prune columns and push filters early — check `explain`.
2. Equi joins map to BHJ / SHJ / SMJ; inequi often explodes (BNLJ / cartesian).
3. Rewrite inequi as equi + residual filter when possible.